# Sales Variance Analysis

The aim is to explain the sales change through prices and Volume/Mix. Start with each customer and product, then add the results by product, customer, and in total.

## 1. Load the data

Keep the CSV in the same folder as this notebook. The data contains 100 customers, five products, and two years. It is synthetic data, with amounts in EUR and no fixed charges.

In [1]:
import pandas as pd
from IPython.display import display

pd.options.display.float_format = '{:,.2f}'.format
df_sales = pd.read_csv('sales_variance_dataset.csv', encoding='utf-8-sig')
df_sales.head(10)


,Year,Customer,Product,Quantity,Price,Sales
0,2025,Customer_001,Product_A,272,4.60,"1,251.20"
1,2025,Customer_001,Product_B,243,7.58,"1,841.94"
2,2025,Customer_001,Product_C,76,11.04,839.04
3,2025,Customer_001,Product_D,406,16.89,"6,857.34"
4,2025,Customer_001,Product_E,363,23.00,"8,349.00"
5,2025,Customer_002,Product_A,479,5.20,"2,490.80"
6,2025,Customer_002,Product_B,351,8.16,"2,864.16"
7,2025,Customer_002,Product_C,916,12.36,"11,321.76"
8,2025,Customer_002,Product_D,874,18.18,"15,889.32"
9,2025,Customer_002,Product_E,943,25.50,"24,046.50"


In [2]:
print('Rows:', len(df_sales))
print('Customers:', df_sales['Customer'].nunique())
print('Products:', df_sales['Product'].nunique())
print('Years:', sorted(df_sales['Year'].unique().tolist()))


Rows: 1000
Customers: 100
Products: 5
Years: [2025, 2026]


## 2. Compare the two years

Put the 2025 and 2026 records side by side, matching on customer and product. All pairs in this dataset are established business and appear in both years.

In [3]:
previous_year = 2025
current_year = 2026
columns = ['Customer', 'Product', 'Quantity', 'Price', 'Sales']

# Each customer-product pair must have one record in each year.
if previous_year == current_year:
    raise ValueError('Choose two different years.')
if df_sales.duplicated(['Year', 'Customer', 'Product']).any():
    raise ValueError('There is more than one record for a customer-product pair in the same year.')

previous = df_sales.loc[df_sales['Year'] == previous_year, columns]
current = df_sales.loc[df_sales['Year'] == current_year, columns]
comparison = previous.merge(
    current, on=['Customer', 'Product'], how='outer',
    suffixes=('_Previous', '_Current'), validate='one_to_one'
)
if comparison.empty or comparison.isna().any().any():
    raise ValueError('Each customer-product pair needs complete data for both years.')

print('Customer-product pairs:', len(comparison))


Customer-product pairs: 500


## 3. Calculate the price impact

Use the change in price and the previous quantity:

$$\text{Price impact}=(P_{2026}-P_{2025})\times Q_{2025}$$

For example, a price increase from EUR 10 to EUR 11 on a previous quantity of 100 units gives a EUR 100 price impact. Keeping quantity at its old level isolates the price change. Positive values add to revenue; negative values reduce it.

In [4]:
comparison['Price_Change'] = (
    comparison['Price_Current'] - comparison['Price_Previous']
).round(2)
comparison['Price_Impact'] = (
    comparison['Price_Change'] * comparison['Quantity_Previous']
).round(2)

detail_columns = ['Customer', 'Product', 'Quantity_Previous',
                  'Price_Previous', 'Price_Current', 'Price_Change', 'Price_Impact']
comparison[detail_columns].head(10)


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
1,Customer_001,Product_B,243,7.58,6.97,-0.61,-148.23
2,Customer_001,Product_C,76,11.04,11.81,0.77,58.52
3,Customer_001,Product_D,406,16.89,15.88,-1.01,-410.06
4,Customer_001,Product_E,363,23.00,26.91,3.91,"1,419.33"
5,Customer_002,Product_A,479,5.20,5.36,0.16,76.64
6,Customer_002,Product_B,351,8.16,7.51,-0.65,-228.15
7,Customer_002,Product_C,916,12.36,14.58,2.22,"2,033.52"
8,Customer_002,Product_D,874,18.18,17.45,-0.73,-638.02
9,Customer_002,Product_E,943,25.50,27.29,1.79,"1,687.97"


## 4. Calculate the Volume/Mix impact

Multiply the quantity change by the current price:

$$\text{Volume/Mix impact}=(Q_{2026}-Q_{2025})\times P_{2026}$$

For example, going from 100 to 120 units at a current price of EUR 11 gives $20\times11=220$ euros. We use the current price, so this includes the price change on the extra units. Together with the price impact, it explains the full sales change.

A positive value adds to revenue; a negative value reduces it. Across customers and products, the calculation includes changes in sales mix. It does not separate a pure volume effect from a pure mix effect. All pairs here are established business; new and lost business are outside this dataset.

In [5]:
comparison['Quantity_Change'] = (
    comparison['Quantity_Current'] - comparison['Quantity_Previous']
)
comparison['Volume_Mix_Impact'] = (
    comparison['Quantity_Change'] * comparison['Price_Current']
).round(2)
comparison['Sales_Change'] = (
    comparison['Sales_Current'] - comparison['Sales_Previous']
).round(2)

volume_columns = ['Customer', 'Product', 'Quantity_Previous', 'Quantity_Current',
                  'Quantity_Change', 'Price_Current', 'Volume_Mix_Impact']
comparison[volume_columns].head(10)


,Customer,Product,Quantity_Previous,Quantity_Current,Quantity_Change,Price_Current,Volume_Mix_Impact
0,Customer_001,Product_A,272,272,0,4.60,0.00
1,Customer_001,Product_B,243,323,80,6.97,557.60
2,Customer_001,Product_C,76,80,4,11.81,47.24
3,Customer_001,Product_D,406,406,0,15.88,0.00
4,Customer_001,Product_E,363,454,91,26.91,"2,448.81"
5,Customer_002,Product_A,479,508,29,5.36,155.44
6,Customer_002,Product_B,351,446,95,7.51,713.45
7,Customer_002,Product_C,916,797,-119,14.58,"-1,735.02"
8,Customer_002,Product_D,874,883,9,17.45,157.05
9,Customer_002,Product_E,943,1113,170,27.29,"4,639.30"


## 5. Add up the results

Add the customer-product contributions by product, by customer, and in total. The first ten customers are shown below; `customer_summary` contains the full table.

The two effects must add up to the sales change:

$$\Delta\text{Sales}=\text{Price impact}+\text{Volume/Mix impact}$$

In [6]:
impact_columns = ['Price_Impact', 'Volume_Mix_Impact', 'Sales_Change']
product_summary = comparison.groupby('Product', as_index=False)[impact_columns].sum().round(2)
customer_summary = comparison.groupby('Customer', as_index=False)[impact_columns].sum().round(2)

total_price_impact = round(comparison['Price_Impact'].sum(), 2)
total_volume_mix_impact = round(comparison['Volume_Mix_Impact'].sum(), 2)
total_sales_change = round(comparison['Sales_Change'].sum(), 2)

print(f'Price impact: EUR {total_price_impact:,.2f}')
print(f'Volume/Mix impact: EUR {total_volume_mix_impact:,.2f}')
print(f'Sales change: EUR {total_sales_change:,.2f}')
display(product_summary)
display(customer_summary.head(10))


Price impact: EUR 143,231.87
Volume/Mix impact: EUR 119,063.74
Sales change: EUR 262,295.61


,Product,Price_Impact,Volume_Mix_Impact,Sales_Change
0,Product_A,"5,706.92","18,066.87","23,773.79"
1,Product_B,"-8,227.50","55,076.47","46,848.97"
2,Product_C,"51,566.61","-28,228.60","23,338.01"
3,Product_D,"4,994.35","10,767.53","15,761.88"
4,Product_E,"89,191.49","63,381.47","152,572.96"


,Customer,Price_Impact,Volume_Mix_Impact,Sales_Change
0,Customer_001,919.56,"3,053.65","3,973.21"
1,Customer_002,"2,931.96","3,930.22","6,862.18"
2,Customer_003,"2,303.94","4,492.46","6,796.40"
3,Customer_004,210.13,"1,914.33","2,124.46"
4,Customer_005,"1,335.39","3,500.04","4,835.43"
5,Customer_006,"1,276.68","2,806.36","4,083.04"
6,Customer_007,"1,670.35","-8,455.28","-6,784.93"
7,Customer_008,"1,623.64","-1,128.56",495.08
8,Customer_009,-256.19,"-2,352.36","-2,608.55"
9,Customer_010,"1,290.92","4,275.87","5,566.79"


## 6. Choose a product or customer

The same filter is used for both effects. `show_volume_mix_impact(...)` shows the quantity calculation, and `show_impacts(...)` shows both effects together. The earlier price functions still work. Names can be entered in upper or lower case.

In [7]:
def sales_details(product=None, customer=None):
    result = comparison
    if product is not None:
        name = str(product).strip().lower()
        result = result[result['Product'].str.lower() == name]
    if customer is not None:
        name = str(customer).strip().lower()
        result = result[result['Customer'].str.lower() == name]
    if result.empty:
        raise ValueError('No data found. Check the product or customer name.')
    return result.copy()


def price_impact_details(product=None, customer=None):
    return sales_details(product, customer)[detail_columns]


def volume_mix_impact_details(product=None, customer=None):
    return sales_details(product, customer)[volume_columns]


def price_impact(product=None, customer=None):
    return round(sales_details(product, customer)['Price_Impact'].sum(), 2)


def volume_mix_impact(product=None, customer=None):
    return round(sales_details(product, customer)['Volume_Mix_Impact'].sum(), 2)


def show_price_impact(product=None, customer=None):
    print(f'Price impact: EUR {price_impact(product, customer):,.2f}')
    if product is None and customer is None:
        display(product_summary[['Product', 'Price_Impact']])
    else:
        display(price_impact_details(product, customer))


def show_volume_mix_impact(product=None, customer=None):
    print(f'Volume/Mix impact: EUR {volume_mix_impact(product, customer):,.2f}')
    if product is None and customer is None:
        display(product_summary[['Product', 'Volume_Mix_Impact']])
    else:
        display(volume_mix_impact_details(product, customer))


def show_impacts(product=None, customer=None):
    result = sales_details(product, customer)
    print(f'Price impact: EUR {price_impact(product, customer):,.2f}')
    print(f'Volume/Mix impact: EUR {volume_mix_impact(product, customer):,.2f}')
    print(f'Sales change: EUR {result["Sales_Change"].sum():,.2f}')
    if product is None and customer is None:
        display(product_summary)
    else:
        display(result[['Customer', 'Product'] + impact_columns])


### Product

Change the product name to see its Volume/Mix impact across customers.

In [8]:
selected_product = "Product_A"
show_volume_mix_impact(product=selected_product)

Volume/Mix impact: EUR 18,066.87


,Customer,Product,Quantity_Previous,Quantity_Current,Quantity_Change,Price_Current,Volume_Mix_Impact
0,Customer_001,Product_A,272,272,0,4.60,0.00
5,Customer_002,Product_A,479,508,29,5.36,155.44
10,Customer_003,Product_A,297,428,131,4.44,581.64
15,Customer_004,Product_A,141,182,41,5.15,211.15
20,Customer_005,Product_A,283,328,45,4.90,220.50
...,...,...,...,...,...,...,...
475,Customer_096,Product_A,241,202,-39,4.70,-183.30
480,Customer_097,Product_A,319,392,73,4.65,339.45
485,Customer_098,Product_A,462,587,125,4.75,593.75
490,Customer_099,Product_A,512,538,26,5.00,130.00


### Customer

The five product contributions add up to the customer's Volume/Mix impact.

In [9]:
selected_customer = "Customer_001"
show_volume_mix_impact(customer=selected_customer)

Volume/Mix impact: EUR 3,053.65


,Customer,Product,Quantity_Previous,Quantity_Current,Quantity_Change,Price_Current,Volume_Mix_Impact
0,Customer_001,Product_A,272,272,0,4.60,0.00
1,Customer_001,Product_B,243,323,80,6.97,557.60
2,Customer_001,Product_C,76,80,4,11.81,47.24
3,Customer_001,Product_D,406,406,0,15.88,0.00
4,Customer_001,Product_E,363,454,91,26.91,"2,448.81"


### Overall total

Without a filter, the function adds all customer-product impacts. Supply both filters for a single pair.

In [10]:
show_impacts()

# Other examples:
# show_price_impact(product="Product_A")
# show_impacts(customer="Customer_001")
# show_volume_mix_impact(product="Product_B", customer="Customer_001")

Price impact: EUR 143,231.87
Volume/Mix impact: EUR 119,063.74
Sales change: EUR 262,295.61


,Product,Price_Impact,Volume_Mix_Impact,Sales_Change
0,Product_A,"5,706.92","18,066.87","23,773.79"
1,Product_B,"-8,227.50","55,076.47","46,848.97"
2,Product_C,"51,566.61","-28,228.60","23,338.01"
3,Product_D,"4,994.35","10,767.53","15,761.88"
4,Product_E,"89,191.49","63,381.47","152,572.96"


## 7. Use dropdowns

Choose a product or customer to see both effects. Leave both at All for the total. Choosing both gives one customer-product pair. If widgets are unavailable, use the example cells above.

In [11]:
try:
    import ipywidgets as widgets
except ImportError:
    print('Use the example cells above if widgets are not installed.')
else:
    widgets.interact(
        show_impacts,
        product=widgets.Dropdown(
            options=[('All products', None)] + [(p, p) for p in product_summary['Product']]
        ),
        customer=widgets.Dropdown(
            options=[('All customers', None)] + [(c, c) for c in customer_summary['Customer']]
        )
    )


interactive(children=(Dropdown(description='product', options=(('All products', None), ('Product_A', 'Product_…

## 8. Check the totals

Check that Price Impact + Volume/Mix equals the sales change for every pair, product, and customer. Both summary tables must also add up to the overall totals.

In [12]:
# The effects must explain the sales change for each customer-product pair.
combined_impact = (comparison['Price_Impact'] + comparison['Volume_Mix_Impact']).round(2)
assert (combined_impact == comparison['Sales_Change']).all()

# The same identity must hold for each product and customer.
for summary in [product_summary, customer_summary]:
    combined = (summary['Price_Impact'] + summary['Volume_Mix_Impact']).round(2)
    assert (combined == summary['Sales_Change']).all()
    for column in impact_columns:
        assert round(summary[column].sum(), 2) == round(comparison[column].sum(), 2)

assert round(total_price_impact + total_volume_mix_impact, 2) == total_sales_change
print('Price impact + Volume/Mix impact = sales change at every level.')


Price impact + Volume/Mix impact = sales change at every level.
